# Data Ingestion – Human-Curated Chunks → Qdrant

Ingests **human-curated chunks** into the Qdrant collection `method_human_chunks_hybrid`.

This single collection is used by **both** Basic RAG and Our RAG pipelines (controlled experiment).

| Item | Value |
|---|---|
| Collection | `method_human_chunks_hybrid` |
| Chunks | 155 (manually segmented by team) |
| Embeddings | Dense: `text-embedding-3-small` + Sparse: BM25 |
| Retrieval mode | Hybrid |

### When to re-run
- Human-curated chunks updated
- Embedding model changed
- Qdrant data lost

In [ ]:
# Cell 1 – Setup
import json
import pandas as pd
from langchain_core.documents import Document
from langchain_qdrant import RetrievalMode

from config import settings, RAG_COLLECTION
from components.vector_store import get_vector_store, delete_vector_store

print(f"Target collection: {RAG_COLLECTION}")
print("Setup OK")

In [ ]:
# Cell 2 – Load human-curated chunks
corpus_df = pd.read_csv("dataset/corpus.csv")
human_chunks_df = pd.read_csv("dataset/human_curated_chunks.csv")

# Merge doc name + metadata from corpus
human_chunks_df = human_chunks_df.merge(
    corpus_df[["id", "name", "document_metadata"]].rename(columns={"id": "document_id"}),
    on="document_id",
    how="left",
)

# Convert to LangChain Documents
documents = []
for _, row in human_chunks_df.iterrows():
    doc = Document(
        page_content=row["name"] + "\n" + row["text"],
        metadata={
            "document_id": row["document_id"],
            "additional_metadata": json.loads(row["document_metadata"]),
        },
    )
    documents.append(doc)

print(f"Loaded {len(documents)} human-curated chunks from {len(corpus_df)} source documents")

In [ ]:
# Cell 3 – (OPTIONAL) Delete existing collection before re-ingesting
# Uncomment ONLY if you want to start fresh

# delete_vector_store(RAG_COLLECTION)

In [ ]:
# Cell 4 – Ingest into Qdrant
vs = get_vector_store(
    mode=RetrievalMode.HYBRID,
    collection_name=RAG_COLLECTION,
)
vs.add_documents(documents)
print(f"Ingested {len(documents)} chunks into '{RAG_COLLECTION}'")

In [ ]:
# Cell 5 – Verify
from qdrant_client import QdrantClient

client = QdrantClient(url=settings.qdrant_url)
info = client.get_collection(RAG_COLLECTION)
print(f"{RAG_COLLECTION}: {info.points_count} points")